# Gold Layer: Star Schema

Builds a star schema from the Silver table: a fact table with consumption, 
production, and price metrics, and a time dimension table with derived 
calendar attributes for reporting.

**Input:** electricity_project.silver.electricity_combined<br>
**Output:** electricity_project.gold.fact_electricity, 
electricity_project.gold.dim_time

In [0]:
from pyspark.sql import functions as F

Creates the gold schema.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS electricity_project.gold")

DataFrame[]

Reads the Silver table as the starting point 
for building the fact and dimension tables.

In [0]:
silver_df = spark.table("electricity_project.silver.electricity_combined")

Creates the fact table by selecting only the timestamp key and the three 
numeric metrics, leaving calendar attributes to the time dimension.

In [0]:
fact_electricity_df = silver_df.select(
    "timestamp",
    "consumption_mw",
    "production_mw",
    "price_eur_mwh"
)

Builds the time dimension from unique timestamps, deriving calendar 
attributes (date, hour, weekday, month, weekend flag) from the Finland 
local time, so reports reflect actual local patterns.

In [0]:
dim_time_df = (
    silver_df.select("timestamp", "timestamp_local").distinct()
    .withColumn("date", F.to_date("timestamp_local"))
    .withColumn("hour", F.hour("timestamp_local"))
    .withColumn("weekday_name", F.date_format("timestamp_local", "EEEE"))
    .withColumn("month", F.month("timestamp_local"))
    .withColumn("is_weekend", F.dayofweek("timestamp_local").isin([1, 7]))
)

Adds a season column based on month, using the standard meteorological 
season boundaries (Dec-Feb winter, etc.) rather than astronomical dates.

In [0]:
dim_time_df = dim_time_df.withColumn(
    "season",
    F.when(F.col("month").isin([12, 1, 2]), "Winter")
     .when(F.col("month").isin([3, 4, 5]), "Spring")
     .when(F.col("month").isin([6, 7, 8]), "Summer")
     .otherwise("Autumn")
)

In [0]:
display(dim_time_df.limit(5))

timestamp,timestamp_local,date,hour,weekday_name,month,is_weekend,season
2026-08-31T23:45:00.000Z,2026-09-01T02:45:00.000Z,2026-09-01,2,Tuesday,9,false,Autumn
2026-08-31T23:30:00.000Z,2026-09-01T02:30:00.000Z,2026-09-01,2,Tuesday,9,false,Autumn
2026-08-31T23:15:00.000Z,2026-09-01T02:15:00.000Z,2026-09-01,2,Tuesday,9,false,Autumn
2026-08-31T23:00:00.000Z,2026-09-01T02:00:00.000Z,2026-09-01,2,Tuesday,9,false,Autumn
2026-08-31T22:45:00.000Z,2026-09-01T01:45:00.000Z,2026-09-01,1,Tuesday,9,false,Autumn


Writes the fact and time dimension tables to the gold schema, completing 
the star schema.

In [0]:
fact_electricity_df.write.format("delta").mode("overwrite").saveAsTable("electricity_project.gold.fact_electricity")
dim_time_df.write.format("delta").mode("overwrite").saveAsTable("electricity_project.gold.dim_time")

Sanity checks.

In [0]:
fact_df = spark.table("electricity_project.gold.fact_electricity")
dim_df = spark.table("electricity_project.gold.dim_time")

print(fact_df.count(), dim_df.count())
print(fact_df.count() == dim_df.count())

32167 32167
True
